# CURE-Rec — complete notebook execution

This notebook is the **single entry point** for the complete Milestone 1 workflow. Running it in order performs every implemented stage:

1. fetch/load external recommendation data;
2. standardize and audit the evidence level;
3. run all registered CPU recommender models;
4. generate external-data analysis assets;
5. execute all CURE-Sim scenarios and all 64 intervention coalitions;
6. compute exact Shapley values, interaction regions, feasibility sensitivity, and direct robust policy selection;
7. generate every numbered CURE-Sim paper asset, logs, manifests, and decision card.

The external-data stage tests data and recommender-model logic. The CURE-Sim stage is the oracle causal benchmark; the notebook does not misrepresent ordinary ratings data as long-horizon policy-intervention evidence.

## 1. Setup

Install once from `paper-ideas/CURE-Rec/code/`:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[dev]'
jupyter lab notebooks/00_cure_rec_quickstart.ipynb
```

In [25]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook from the CURE-Rec code directory or repository root.')

# VS Code kernels are long-lived. Always force this notebook to use the current
# repository source, not a stale cure_rec module imported before a git pull.
sys.path[:] = [str(ROOT), *[entry for entry in sys.path if entry != str(ROOT)]]
for module_name in list(sys.modules):
    if module_name == 'cure_rec' or module_name.startswith('cure_rec.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

from cure_rec.config import load_settings
from cure_rec.experiments import run_seed_sweep
from cure_rec.regimes import run_regime_suite
from cure_rec.observability import RunLogger
from cure_rec.data import DatasetLoadResult, load_dataset
from cure_rec.workflow import run_full_workflow

import cure_rec.data as data_layer
assert hasattr(data_layer, 'DatasetLoadResult'), f'Stale data module loaded: {data_layer.__file__}'
print('Project root:', ROOT)
print('Data module:', data_layer.__file__)
print('CURE-Rec data layer: current')


Project root: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code
Data module: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/cure_rec/data.py
CURE-Rec data layer: current


## 2. Configure the entire run

`quick` is designed for interactive use. `full` uses the larger CURE-Sim configuration. The external-data fetch is explicit and visible: set `FETCH_IF_MISSING = False` when using already downloaded local data.

In [26]:
# CURE-Sim configuration
CURE_MODE = 'full'  # quick | full
config_name = 'curesim_quickstart.yaml' if CURE_MODE == 'quick' else 'curesim_full.yaml'
settings = load_settings(ROOT / 'configs' / config_name)

# External-data and registered model configuration
PUBLIC_DATASET = 'movielens_1m'  # movielens_1m | coat | yahoo_r3 | csv
PUBLIC_SOURCE = ROOT / 'data' / 'raw' / PUBLIC_DATASET
FETCH_IF_MISSING = True  # explicit opt-in network retrieval for MovieLens-1M / Coat
RUN_BPR_MF = True
BPR_UPDATES = 500_000 if CURE_MODE == 'quick' else 1_500_000
MAX_EVAL_USERS = 1_000

print('CURE config:', config_name, '| hash:', settings.config_hash())
print('CURE users/items/horizon:', settings.simulator.n_users, settings.simulator.n_items, settings.simulator.horizon)
print('CURE interventions:', list(settings.interventions.costs))
print('CURE scenarios:', [scenario.name for scenario in settings.scenarios])
print('External dataset:', PUBLIC_DATASET, '| source:', PUBLIC_SOURCE)


CURE config: curesim_full.yaml | hash: 9451b1ec1ac3321e
CURE users/items/horizon: 120 240 12
CURE interventions: ['repeat_cap', 'explore_slot', 'tail_slot', 'diversify', 'novel_slot', 'provider_balance']
CURE scenarios: ['nominal', 'fatigue_stress', 'popularity_stress', 'mixed_stress']
External dataset: movielens_1m | source: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/data/raw/movielens_1m


## 3. Run every implemented stage

This one cell calls the full orchestrator. It **fetches/loads data first**, audits the evidence, runs popularity and BPR-MF baselines when chronology is available, and then runs the full CURE-Sim causal workflow and asset generator.

In [27]:
workflow = run_full_workflow(
    settings,
    dataset=PUBLIC_DATASET,
    source=PUBLIC_SOURCE,
    download=FETCH_IF_MISSING,
    run_bpr=RUN_BPR_MF,
    bpr_updates=BPR_UPDATES,
    max_eval_users=MAX_EVAL_USERS,
)

public_result = workflow.dataset
data_analysis = workflow.analysis
logger = workflow.logger
game = workflow.game
RUN_DIR = workflow.cure_run_dir
decision = workflow.decision

print('External-data analysis run:', data_analysis.run_dir)
print('External-data evidence level:', data_analysis.audit.permitted_claim)
print('CURE-Rec run:', RUN_DIR)
print('Decision:', decision.action)
print('Selected portfolio:', decision.selected_interventions)
print('Worst-case improvement:', round(decision.lower_improvement, 5))


2026-08-04 17:01:12,511 | INFO | run_started | {"config_hash": "9451b1ec1ac3321e", "run_id": "curesim-full-20260804T160112Z-e876dee3"}
2026-08-04 17:01:12,511 | INFO | exact_game_started | {}
2026-08-04 17:01:12,512 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-04 17:03:47,741 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13323832372399413, "scenario": "nominal", "shapley_efficiency_gap": 1.3877787807814457e-16}
2026-08-04 17:03:47,741 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-04 17:07:30,875 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13544545518030754, "scenario": "fatigue_stress", "shapley_efficiency_gap": 0.0}
2026-08-04 17:07:30,876 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-04 17:10:13,038 | INFO | scenario_game_completed | {"gran

## 4. Inspect loaded data, audit logic, and every registered baseline model

These results describe the external interaction data. They are intentionally separate from the CURE-Sim causal results below.

In [28]:
print('Loader metadata:')
print(public_result.metadata)
print('Audit notes:', *data_analysis.audit.notes, sep='\n- ')
display(data_analysis.summary)
display(data_analysis.model_metrics)

print('External-data assets:')
for path in sorted((data_analysis.run_dir / 'tables').glob('*.csv')):
    print('-', path.name)
for path in sorted((data_analysis.run_dir / 'figures').glob('*.png')):
    print('-', path.name)


Loader metadata:
{'ratings_path': '/Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/data/raw/movielens_1m/ml-1m/ratings.dat', 'rows': 1000209, 'has_exposure_log': False}
Audit notes:
- Missing logged slate/propensity fields; do not make offline causal-policy claims.


,dataset,users,items,interactions,positive_interactions,positive_rate,density,timestamps_available
0,movielens_1m,6040,3706,1000209,575281,0.575161,0.044684,True


,model,evaluated_users,recall_at_k,ndcg_at_k,hit_rate_at_k
0,popularity,1000,0.049,0.025520,0.049
1,bpr_mf,1000,0.050,0.023393,0.050


External-data assets:
- data_table_bpr_loss.csv
- data_table_bpr_validation.csv
- data_table_item_activity.csv
- data_table_model_metrics.csv
- data_table_summary.csv
- data_table_user_activity.csv
- data_figure_activity_distributions.png
- data_figure_bpr_training.png
- data_figure_model_metrics.png


## 5. Inspect the complete CURE-Rec causal game

The game uses all six interventions, exact coalition values, exact Shapley contributions, feasibility-aware semivalue sensitivity, and Grabisch–Roubens pairwise interactions. Direct robust improvement—not a sum of Shapley lower endpoints—selects the portfolio.

In [29]:
display(game.regions.sort_values('phi_mean', ascending=False))
display(game.interaction_table.sort_values('interaction_mean', ascending=False))

coalitions = game.coalition_table.groupby('mask', as_index=False).agg(
    lower_improvement=('improvement', 'min'),
    upper_improvement=('improvement', 'max'),
    cost=('cost', 'first'),
    interventions=('active_interventions', 'first'),
).sort_values('lower_improvement', ascending=False)
display(coalitions.head(12))


,intervention,phi_lower,phi_upper,phi_mean,psi_feasible_lower,psi_feasible_upper,phi_psi_sign_agree
0,repeat_cap,0.265615,0.282537,0.273900,0.264458,0.281175,True
3,diversify,-0.060068,-0.058732,-0.059446,-0.061531,-0.059862,True
2,tail_slot,-0.068889,-0.066982,-0.067716,-0.069026,-0.067058,True
4,novel_slot,-0.083828,-0.079994,-0.081727,-0.083546,-0.079884,True
1,explore_slot,-0.087903,-0.087077,-0.087454,-0.088215,-0.087239,True
5,provider_balance,-0.119245,-0.117743,-0.118261,-0.120200,-0.118603,True


,intervention_i,intervention_j,interaction_lower,interaction_upper,interaction_mean
13,diversify,provider_balance,0.001772,0.004749,0.003069
14,novel_slot,provider_balance,0.001114,0.004812,0.002574
2,repeat_cap,diversify,0.000410,0.002658,0.001198
12,diversify,novel_slot,0.000053,0.001850,0.001092
4,repeat_cap,provider_balance,-0.002106,0.002614,0.000638
5,explore_slot,tail_slot,-0.001099,0.000478,-0.000249
7,explore_slot,novel_slot,-0.000779,0.000627,-0.000282
10,tail_slot,novel_slot,-0.004616,-0.002083,-0.002858
6,explore_slot,diversify,-0.004400,-0.002136,-0.003046
8,explore_slot,provider_balance,-0.003468,-0.003015,-0.003264


,mask,lower_improvement,upper_improvement,cost,interventions
1,1,0.294495,0.312545,0.05,repeat_cap
9,9,0.234653,0.252562,0.11,repeat_cap;diversify
5,5,0.220450,0.234863,0.13,repeat_cap;tail_slot
17,17,0.199789,0.218967,0.13,repeat_cap;novel_slot
3,3,0.195329,0.210596,0.15,repeat_cap;explore_slot
33,33,0.174731,0.192387,0.17,repeat_cap;provider_balance
13,13,0.159495,0.174496,0.19,repeat_cap;tail_slot;diversify
25,25,0.145149,0.163012,0.19,repeat_cap;diversify;novel_slot
11,11,0.134867,0.151341,0.21,repeat_cap;explore_slot;diversify
21,21,0.126669,0.140984,0.21,repeat_cap;tail_slot;novel_slot


## 6. Inspect all numbered paper assets, logs, and manifests

Every CURE-Sim run generates Tables 1–8, Figures 1–8, an asset registry, per-coalition manifests, JSONL events, raw coalition values, and a deployment/explanation decision card.

In [30]:
asset_manifest = json.loads((RUN_DIR / 'artifacts' / 'asset_manifest.json').read_text())
asset_registry = pd.DataFrame(asset_manifest)
display(asset_registry)

print('Generated CURE tables:')
for path in sorted((RUN_DIR / 'tables').glob('*.csv')):
    print('-', path.name)
print('\nGenerated CURE figures:')
for path in sorted((RUN_DIR / 'figures').glob('*.png')):
    print('-', path.name)

events = pd.DataFrame([json.loads(line) for line in (RUN_DIR / 'logs' / 'events.jsonl').read_text().splitlines()])
display(events[['timestamp_utc', 'event']].tail(20))

decision_card = json.loads((RUN_DIR / 'artifacts' / 'explanation_card.json').read_text())
decision_card


,exists,id,path,purpose,scope
0,True,Table 1,tables/table_01_asset_registry.csv,"Asset provenance, scope, and readiness",generated
1,True,Table 2,tables/table_02_benchmark_configuration.csv,CURE-Sim and policy configuration,generated
2,True,Table 3,tables/table_03_attribution_regions.csv,Full-game Shapley and feasibility-aware semiva...,generated
3,True,Table 4,tables/table_04_uncertainty_summary.csv,Scenario uncertainty widths and attribution signs,generated
4,True,Table 5,tables/table_05_portfolio_decision.csv,Robust selected portfolio and constraint diagn...,generated
5,True,Table 6,tables/table_06_long_term_tradeoffs.csv,Base versus selected policy outcomes by scenario,generated
6,True,Table 7,tables/table_07_selection_comparison.csv,"Base, best-single, full, and robust portfolio ...",generated
7,True,Table 8,tables/table_08_runtime_summary.csv,Coalition evaluation runtime by scenario and c...,generated
8,True,Figure 1,figures/figure_01_framework.png,CURE-Rec execution flow,generated
9,True,Figure 2,figures/figure_02_shapley_regions.png,Shapley regions and selected interventions,generated


Generated CURE tables:
- coalition_values.csv
- interaction_regions.csv
- shapley_regions.csv
- table_01_asset_registry.csv
- table_02_benchmark_configuration.csv
- table_03_attribution_regions.csv
- table_04_uncertainty_summary.csv
- table_05_portfolio_decision.csv
- table_06_long_term_tradeoffs.csv
- table_07_selection_comparison.csv
- table_08_runtime_summary.csv

Generated CURE figures:
- figure_01_framework.png
- figure_02_shapley_regions.png
- figure_03_uncertainty_widths.png
- figure_04_interaction_heatmap.png
- figure_05_trajectory_comparison.png
- figure_06_decision_card.png
- figure_07_runtime_by_cardinality.png
- figure_08_scenario_sensitivity.png


,timestamp_utc,event
537,2026-08-04T16:13:08.053889+00:00,portfolio_rejected
538,2026-08-04T16:13:08.054285+00:00,portfolio_rejected
539,2026-08-04T16:13:08.054577+00:00,portfolio_rejected
540,2026-08-04T16:13:08.054869+00:00,portfolio_rejected
541,2026-08-04T16:13:08.055238+00:00,portfolio_rejected
542,2026-08-04T16:13:08.055418+00:00,portfolio_rejected
543,2026-08-04T16:13:08.055775+00:00,portfolio_rejected
544,2026-08-04T16:13:08.056071+00:00,portfolio_rejected
545,2026-08-04T16:13:08.056445+00:00,portfolio_rejected
546,2026-08-04T16:13:08.056715+00:00,portfolio_rejected


{'decision': {'action': 'repair_selected',
  'base_feasible': False,
  'cost': 0.05,
  'fatigue_upper': 0.0,
  'feasible': True,
  'lower_improvement': 0.2944949756908204,
  'mode': 'repair',
  'provider_disparity_upper': 0.2442746913580247,
  'reason': 'Base policy violates robust constraints; selected the best feasible repair by maximin improvement.',
  'relevance_delta_lower': -0.041477531911315535,
  'selected_interventions': ['repeat_cap'],
  'selected_mask': 1,
  'status': 'repair_selected',
  'upper_improvement': 0.3125451363200903},
 'interpretation': {'improvement_mode': 'A feasible base policy may be retained when no portfolio robustly improves it.',
  'positive_certificate': 'phi_lower > 0 means positive order-averaged marginal contribution across configured scenarios.',
  'repair_mode': 'An infeasible base policy cannot be retained; select a feasible repair or certify no feasible portfolio.',
  'selection_rule': 'Portfolio selection used direct robust improvement, not summe

## 7. Optional: paired multi-seed stabilization run

After validating the full single-seed run, set `RUN_SEED_SWEEP = True` to run paired CURE-Sim seeds with common random numbers within each seed. Start with five seeds for stabilization; use 20–30 seeds for final scientific comparisons.

In [32]:
RUN_SEED_SWEEP = True
SEEDS = [42, 43, 44, 45, 46]

if RUN_SEED_SWEEP:
    seed_sweep = run_seed_sweep(settings, SEEDS)
    print("Seed-sweep assets:", seed_sweep.run_dir)
    display(seed_sweep.decisions)
    display(seed_sweep.attributions.groupby("intervention", as_index=False).agg(
        phi_mean=("phi_mean", "mean"),
        phi_std=("phi_mean", "std"),
        positive_rate=("phi_lower", lambda x: float((x > 0).mean())),
    ))
else:
    print("Seed sweep disabled. Set RUN_SEED_SWEEP = True after inspecting the single-seed full run.")


2026-08-04 17:28:32,436 | INFO | run_started | {"config_hash": "519261a28790f894", "run_id": "curesim-full-seed-42-20260804T162832Z-83614aa8"}
2026-08-04 17:28:32,437 | INFO | exact_game_started | {}
2026-08-04 17:28:32,439 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-04 17:31:16,132 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13710546477148466, "scenario": "nominal", "shapley_efficiency_gap": 0.0}
2026-08-04 17:31:16,133 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-04 17:34:04,765 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13194331906701645, "scenario": "fatigue_stress", "shapley_efficiency_gap": 2.7755575615628914e-17}
2026-08-04 17:34:04,766 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-04 17:36:59,467 | INFO | scenario_game_completed 

,seed,cure_run_dir,mode,status,base_feasible,selected_mask,selected_interventions,lower_improvement,upper_improvement,cost,relevance_delta_lower,provider_disparity_upper,fatigue_upper,feasible,reason,action
0,42,runs/seed-sweep-20260804T162832Z/runs/curesim-...,repair,repair_selected,False,1,"(repeat_cap,)",0.298514,0.313069,0.05,-0.044137,0.225872,0.0,True,Base policy violates robust constraints; selec...,repair_selected
1,43,runs/seed-sweep-20260804T162832Z/runs/curesim-...,improvement,improve_selected,True,1,"(repeat_cap,)",0.296202,0.310991,0.05,-0.029899,0.202785,0.0,True,Base policy is feasible; selected the exact ma...,improve_selected
2,44,runs/seed-sweep-20260804T162832Z/runs/curesim-...,repair,repair_selected,False,1,"(repeat_cap,)",0.292761,0.305775,0.05,-0.043151,0.219900,0.0,True,Base policy violates robust constraints; selec...,repair_selected
3,45,runs/seed-sweep-20260804T162832Z/runs/curesim-...,repair,repair_selected,False,1,"(repeat_cap,)",0.293929,0.313733,0.05,-0.037734,0.256836,0.0,True,Base policy violates robust constraints; selec...,repair_selected
4,46,runs/seed-sweep-20260804T162832Z/runs/curesim-...,repair,repair_selected,False,1,"(repeat_cap,)",0.293002,0.310941,0.05,-0.036126,0.219884,0.0,True,Base policy violates robust constraints; selec...,repair_selected


,intervention,phi_mean,phi_std,positive_rate
0,diversify,-0.059437,0.000205,0.0
1,explore_slot,-0.087499,0.000170,0.0
2,novel_slot,-0.082152,0.000694,0.0
3,provider_balance,-0.118471,0.000641,0.0
4,repeat_cap,0.273885,0.002306,1.0
5,tail_slot,-0.067628,0.000357,0.0


## 8. Optional: controlled cooperative-structure benchmark suite

Run this suite before large natural CURE-Sim sweeps. It verifies that the planner and attribution machinery recover known additive, complementary, redundant, antagonistic, delayed-fatigue, repair, and misspecified-ambiguity structures. The values are transparent analytical oracle games, not post-hoc tuned behavioural results.

In [ ]:
RUN_REGIME_SUITE = False

if RUN_REGIME_SUITE:
    regime_logger = RunLogger(settings)
    try:
        regime_suite = run_regime_suite(settings, regime_logger)
        regime_logger.close(status="completed")
    except Exception:
        regime_logger.close(status="failed")
        raise
    print("Regime-suite assets:", regime_suite.run_dir)
    display(regime_suite.summary)
    display(regime_suite.attribution_recovery.groupby("regime", as_index=False).agg(
        shapley_mae=("absolute_error", "mean"),
        sign_accuracy=("sign_correct", "mean"),
    ))
else:
    print("Regime suite disabled. Set RUN_REGIME_SUITE = True before final multi-seed CURE-Sim runs.")


## 7. Evidence and runtime notes

- MovieLens, Coat, Yahoo! R3, and generic ratings CSVs are loaded and audited before baseline-model analysis. Their audit result controls the permitted scientific claim.
- CURE-Sim is the full causal/oracle environment in this milestone.
- Real long-horizon policy/OPE assets remain gated until an audited slate-policy log and sequential estimator are implemented.
- For a larger CURE-Sim run, change `CURE_MODE = 'full'` and rerun this notebook from the top.
- The same workflow is available from the terminal via `cure-rec full-run`.